# Destiny Explained Studio 0.8.2 — Cinematic Voxel Environments

One PNG, with live rendering logs and a configurable timeout. Select the procedural **Riis — Before the Whirlwind** showcase, Draft/Balanced/Cinematic and resolution percentage. CPU is supported; Cycles checks actual GPU devices, with CPU fallback. No video or external visual assets.

Avant fusion : REF `feat/cinematic-voxel-environments-0.8.2`. Après fusion : `main`. Interface des formulaires en français ; aucun changement au pipeline Remotion/Kokoro. Riis est une reconstitution artistique, pas une carte canonique. Télécharger le ZIP avant de fermer Colab.


In [ ]:
# 1 — Installer/détecter les outils (exécuter une fois)
import os, sys, json, shutil, subprocess, platform, urllib.request, hashlib, tarfile
from pathlib import Path
assert sys.version_info >= (3, 11), "Utiliser Python 3.11 ou plus récent."
REF = "main" # @param ["main", "feat/cinematic-voxel-environments-0.8.2"]
PROJECT = Path('/content/Destiny-Explained-Studio')
if not PROJECT.exists():
    subprocess.run(['git','clone','--branch',REF,'--single-branch','https://github.com/Alexanst54/Destiny-Explained-Studio.git',str(PROJECT)],check=True)
else:
    if subprocess.check_output(['git','-C',str(PROJECT),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Le checkout contient des modifications : les sauvegarder avant de changer REF.')
    subprocess.run(['git','-C',str(PROJECT),'fetch','origin',f'refs/heads/{REF}:refs/remotes/origin/{REF}'],check=True)
    subprocess.run(['git','-C',str(PROJECT),'switch',REF],check=True)
    subprocess.run(['git','-C',str(PROJECT),'merge','--ff-only',f'origin/{REF}'],check=True)
    if subprocess.check_output(['git','-C',str(PROJECT),'rev-parse','HEAD'],text=True).strip()!=subprocess.check_output(['git','-C',str(PROJECT),'rev-parse',f'origin/{REF}'],text=True).strip():
        raise RuntimeError('Commits locaux conservés : utiliser une session propre pour cette REF.')
ROOT = PROJECT/'remotion'
if not (ROOT/'blender/import_scene.py').is_file():
    raise RuntimeError('Cette REF ne contient pas le bridge. Pendant la revue, choisir feat/cinematic-voxel-environments-0.8.2.')
# Node officiel et version verrouillée ; vérifier le SHA publié avant extraction.
version='22.23.4'
existing=shutil.which('node')
if not existing or subprocess.check_output([existing,'-p','process.versions.node'],text=True).strip()!=version:
    arch={'x86_64':'x64','aarch64':'arm64'}.get(platform.machine())
    if not arch: raise RuntimeError('Architecture Node non prise en charge par ce notebook.')
    name=f'node-v{version}-linux-{arch}.tar.xz';base=f'https://nodejs.org/dist/v{version}/'
    cache=Path('/content/destiny-tools');cache.mkdir(exist_ok=True)
    archive=cache/name
    urllib.request.urlretrieve(base+name,archive)
    manifest=urllib.request.urlopen(base+'SHASUMS256.txt').read().decode()
    expected=next(line.split()[0] for line in manifest.splitlines() if line.split()[-1]==name)
    assert hashlib.sha256(archive.read_bytes()).hexdigest()==expected, 'SHA Node incorrect'
    with tarfile.open(archive) as bundle: bundle.extractall(cache,filter='data')
    os.environ['PATH']=str(cache/f'node-v{version}-linux-{arch}'/'bin')+os.pathsep+os.environ['PATH']
NODE=shutil.which('node')
subprocess.run(['npm','ci','--no-audit','--no-fund'],cwd=ROOT,check=True)
BLENDER=shutil.which('blender')
if not BLENDER:
    if os.geteuid()!=0: raise RuntimeError('Installer Blender via le gestionnaire système, puis relancer cette cellule.')
    subprocess.run(['apt-get','-qq','update'],check=True)
    subprocess.run(['apt-get','-qq','install','-y','blender'],check=True)
    BLENDER=shutil.which('blender')
if not BLENDER: raise RuntimeError('Blender non installé.')
print(subprocess.check_output([BLENDER,'--version'],text=True).splitlines()[0])
print('Commit:',subprocess.check_output(['git','-C',str(PROJECT),'rev-parse','HEAD'],text=True).strip())


In [ ]:
# 2 — Ressources observées (la sélection effective sera dans chaque rapport Blender)
cpus=len(os.sched_getaffinity(0)) if hasattr(os,'sched_getaffinity') else os.cpu_count()
print('CPU disponibles:',cpus)
print(next((line for line in Path('/proc/meminfo').read_text().splitlines() if line.startswith('MemTotal:')),'RAM inconnue'))
if shutil.which('nvidia-smi'):
    gpu=subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True,timeout=15)
    print(gpu.stdout.strip() if gpu.returncode==0 else 'GPU non accessible : repli CPU possible.')
else: print('Pas de nvidia-smi : Blender vérifiera ses backends, puis choisira CPU si nécessaire.')


## Choisir une scène depuis Android

`riis-082` : six Éliksni de la frame 210, leurs poses et la caméra restent inchangées ; la ville remplace explicitement le décor minimal. `legacy-081` : ancienne scène pour comparaison. `upload` : un JSON Studio ou snapshot (4 Mio maximum, projet 2 Mio). Un snapshot est déjà figé ; le champ FRAME ne change pas ses poses.

Draft : géométrie réduite / 8 samples ; Balanced : 32 ; Cinematic : 96 et volume atmosphérique borné. Choisir 25 % pour commencer (270×480), 100 % pour 1080×1920. Aucun coût GPU ou durée garantis. Pour un upload, ses réglages d’environnement sont conservés ; les choix DETAIL/LIGHTING/SEED ne s’appliquent qu’au showcase.


In [ ]:
# 3 — Préparer exactement une image
SHOWCASE = "riis-082" # @param ["riis-082", "legacy-081", "upload"]
DETAIL = "balanced" # @param ["draft", "balanced", "cinematic"]
LIGHTING = "dawn" # @param ["dawn", "blue-hour"]
SEED = 42 # @param {type:"integer"}
FRAME = 210 # @param {type:"integer"}
LEGACY_PROFILE = "standard" # @param ["standard", "cinematic"]
SCALE = 25 # @param {type:"slider", min:10, max:100, step:5}
DEVICE = "auto" # @param ["auto", "cpu"]
TIMEOUT_SECONDS = 600 # @param {type:"integer"}
if type(TIMEOUT_SECONDS) is not int or not 10<=TIMEOUT_SECONDS<=7200:raise ValueError('Délai : 10–7200 secondes.')
if type(SCALE) is not int or not 10<=SCALE<=100:raise ValueError('Échelle : 10–100 %.')
import time
OUT=ROOT/'out'/f'blender-colab-{time.time_ns()}';OUT.mkdir(parents=True)
sys.path.insert(0,str(ROOT/'blender'))
from contract import validate
from environments.riis import generate
if SHOWCASE in ['riis-082','legacy-081']:
    data=json.loads((ROOT/'blender/examples/riis-frame-210.json').read_text())
    data['profile']=LEGACY_PROFILE
    if SHOWCASE=='riis-082':data=generate(data,DETAIL,SEED,LIGHTING)
    print('Exemple figé à la frame 210 ; FRAME ne modifie pas ses poses.')
elif SHOWCASE=='upload':
    from google.colab import files
    uploaded=files.upload()
    if len(uploaded)!=1:raise ValueError('Sélectionner exactement un JSON, sans ZIP.')
    payload=next(iter(uploaded.values()))
    if len(payload)>4*1024*1024:raise ValueError('Entrée supérieure à 4 Mio.')
    data=json.loads(payload.decode('utf-8'))
else:raise ValueError('Showcase inconnu.')
if not isinstance(data,dict):raise ValueError('Objet JSON attendu.')
source=OUT/'input.json';source.write_text(json.dumps(data,ensure_ascii=False))
if data.get('format')=='destiny-blender-scene':
    validate(data);snapshots=[source]
else:
    if source.stat().st_size>2*1024*1024:raise ValueError('Projet Studio supérieur à 2 Mio.')
    snapshot=OUT/f'frame-{FRAME:05d}.json'
    subprocess.run([NODE,'--experimental-strip-types','blender/export.mjs',str(source),str(FRAME),str(snapshot),LEGACY_PROFILE],cwd=ROOT,check=True,timeout=90)
    snapshots=[snapshot]
assert len(snapshots)==1, 'Une seule frame, jamais de vidéo.'
print('Snapshot:',snapshots[0].name,'Environnement:',json.loads(snapshots[0].read_text()).get('environment','legacy'))


In [ ]:
# 4 — Progression Blender en direct et rapport de durée
from render_one import render_one
results=[]
for snapshot in snapshots:
    frame=json.loads(snapshot.read_text())['source']['frame']
    output=OUT/f'frame-{frame:05d}.png'
    try:
        benchmark=render_one(snapshot,output,BLENDER,DEVICE,SCALE,max(1,min(cpus or 1,4)),TIMEOUT_SECONDS)
    except Exception as error:
        print('ÉCHEC:',error,'Logs/rapport disponibles dans',OUT)
        print('Exécuter la cellule de téléchargement, puis analyser avant de relancer.')
        raise
    config=benchmark['report']['configuration']
    print('Device effectif:',config['selected'],config['backend'],config['devices'])
    print('Durée processus:',benchmark['wallSeconds'],'s ; PNG:',benchmark['pngBytes'],'octets')
    print('Mémoire RSS Blender:',benchmark['report']['maxProcessRssKiB'],'Kio (hors VRAM/driver).')
    results.append(output)


In [ ]:
# 5 — Prévisualiser et télécharger, même après un échec
from IPython.display import display, Image
from google.colab import files
for output in results:display(Image(filename=str(output),width=270))
archive=shutil.make_archive(str(OUT)+'-results','zip',OUT)
files.download(archive)


## Benchmark et reprise

Pour comparer CPU/GPU : garder le même JSON, seed, détail, SCALE et version de Blender ; rendre une fois avec DEVICE `cpu`, puis `auto` dans un nouveau dossier. Contrôler `configuration.selected/backend/devices`, `wallSeconds`, `maxProcessRssKiB`, dimensions et `pngBytes`. `nvidia-smi` seul ne prouve pas un rendu CUDA. Un timeout arrête le groupe de processus Blender ; les logs et le rapport d’échec sont conservés. Le CPU ne réessaie pas en boucle ; une panne GPU autorise un seul repli CPU dans la limite du délai global.

L’utilisateur a validé le bridge 0.8.1 sur Tesla T4/CUDA dans Colab Pro ; ce n’est pas une mesure de performance 0.8.2. Le notebook 0.8.2 est vérifié localement, sans prétendre avoir exécuté Colab distant ou un nouveau benchmark GPU. OIDN/AgX sont optionnels et se dégradent explicitement selon le build installé. Pas de rendu vidéo.

Reprise : `docs/CINEMATIC-ENVIRONMENTS-PROGRESS.md`. Guide, mesures CPU et limites : `remotion/blender/ENVIRONMENTS.md`.
